<img style="float: center;" src='./assets/jwebbinar.png' width="1000px"/> 

## JWebbinar 49
_June 25, 2026_
## Alignment Introduction 

### Author: Justin Pierel (STScI), Armin Rest (STScI)

**Purpose**:<BR>
This notebook contains a toy example of catalog-based alignment, which is a good primer for the next notebook using real JWST data and a more complicated matching algorithm.


<hr style="border:1px solid gray"> </hr>

# Concept Check
In this section, we'll play with a toy example of how alignment operates and the pitfalls of automatic alignment methods. 

<hr style="border:1px solid gray"> </hr>

In [ ]:
%matplotlib widget

In [ ]:
# If the cell above fails, uncomment the next line and run this cell
# ! pip install ipympl


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import ipywidgets as widgets
from IPython.display import display, clear_output

# Fixed random positions
np.random.seed(42)
max_points = 100
x_all = np.random.uniform(-10, 10, max_points)
y_all = np.random.uniform(-10, 10, max_points)
positions = np.column_stack((x_all, y_all))

def apply_transform_centered(pts, dx, dy, rot_deg):
    """Rotate about the centroid, then shift."""
    center = pts.mean(axis=0)
    pts_centered = pts - center

    theta = np.deg2rad(rot_deg)
    rot_matrix = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

    rotated = pts_centered @ rot_matrix.T
    return rotated + center + np.array([dx, dy])

def make_plot(n1, n2, dx, dy, rot):
    pts1 = positions[:n1]
    pts2 = positions[:n2]

    pts2_shifted = apply_transform_centered(pts2, dx, dy, rot)

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    plt.subplots_adjust(hspace=0.2, wspace=0.3, bottom=0.2, left=0.1)

    for ax in axes:
        ax.set_xlim(-15, 15)
        ax.set_ylim(-15, 15)
        ax.grid(linestyle=":")

    # Panel 1
    axes[0].scatter(pts1[:, 0], pts1[:, 1], marker='*', color='black', label='Target')
    axes[0].scatter(pts2_shifted[:, 0], pts2_shifted[:, 1], marker='*', color='red', label='Reference')
    axes[0].set_aspect('equal', 'box')
    axes[0].legend()
    axes[0].set_xlabel('y')
    axes[0].set_ylabel('x')

    # Panel 2: direct index matching
    min_len = min(n1, n2)
    dx_before = pts2_shifted[:min_len, 0] - pts1[:min_len, 0]

    axes[1].scatter(pts1[:min_len, 1], dx_before, color='blue')
    axes[1].set_xlabel('y')
    axes[1].set_ylabel('dx')
    axes[1].set_title('Correct matching')

    # Panel 3: nearest-neighbor matching
    tree = cKDTree(pts1)
    dists, idxs = tree.query(pts2_shifted[:min_len], k=1)

    dx_after = pts2_shifted[:min_len] - pts1[idxs]

    axes[2].scatter(pts1[idxs, 1], dx_after[:, 0], color='green')
    axes[2].set_xlabel('y')
    axes[2].set_ylabel('dx')
    axes[2].set_title('Naive Matching')

    plt.show()

# Sliders
slider_n1 = widgets.IntSlider(value=50, min=1, max=max_points, step=1, description='N Ref')
slider_n2 = widgets.IntSlider(value=50, min=1, max=max_points, step=1, description='N Targ')
slider_dx = widgets.FloatSlider(value=0.0, min=-10, max=10, step=0.1, description='x offset')
slider_dy = widgets.FloatSlider(value=0.0, min=-10, max=10, step=0.1, description='y offset')
slider_rot = widgets.FloatSlider(value=0.0, min=-45, max=45, step=1, description='Rot. (deg)')

ui = widgets.VBox([
    slider_n1,
    slider_n2,
    slider_dx,
    slider_dy,
    slider_rot
])

out = widgets.interactive_output(make_plot, {
    'n1': slider_n1,
    'n2': slider_n2,
    'dx': slider_dx,
    'dy': slider_dy,
    'rot': slider_rot
})

display(ui, out)

Play with the slider bars above to understand catalog-based alignment for JWST images. The first plot is toy example of an image, where red stars have been identified in the reference image (or from a reference catalog, e.g., Gaia) and black stars have been measured in the target image (i.e., the image to be aligned). When there is no x/y/rotational offset, the target image is perfectly aligned to the reference catalog. The central plot shows y vs. dx, which is flat and centered on 0 when the alignment is perfect, and the third panel is "after correction" comparing the target and reference star locations. See what happens to each plot as you vary the x/y/rotational offsets in particular. 

### Discussion point: What causes major issues for automatic alignment? What are some requirements for identifying if the correction was successful?

### Discussion point: What methods could you use to mitigate the major issues?